# Dimensionality Dynamics — Volume as a Training Signal

The model's class centroid geometry oscillates between 1D (a line), 2D (a ring/plane),
and 3D (a volume) during training. These shifts are expansion/contraction events — moments
when the model opens or closes working space in the representation.

## Two companion metrics

**PR₃** — Participation Ratio of the top 3 eigenvalues:  
```
PR₃ = (f₁ + f₂ + f₃)² / (f₁² + f₂² + f₃²)
```
Range [1, 3]. Measures the *shape* of how variance is distributed within the top 3 dimensions:
- PR₃ ≈ 1 → line (all variance on PC1)
- PR₃ ≈ 2 → ring/plane (variance split across two dimensions)
- PR₃ ≈ 3 → sphere (variance evenly spread across all three)

**f_top3** — Fraction of total variance in the top 3 components:  
```
f_top3 = f₁ + f₂ + f₃
```
Range [0, 1]. Measures whether variance is *concentrated* in the top 3 or *diffusing* into
dimensions 4 and beyond.

Together they trace the geometry: a healthy ring lives at (f_top3 high, PR₃ ≈ 2).
An expansion event moves toward PR₃ ≈ 3. Diffusion into noise moves f_top3 down.

## What to look for

- **Spikes in PR₃** (transient excursions toward 3 or toward 1) = reorganization events
- **Lead/lag across sites** = which component opens the working space first
- **f_top3 behavior during spikes** = does the model expand within top 3, or bleed into noise?
- **Cross-variant comparison** = do healthy/late/degraded models show different PR₃ trajectories?

In [1]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader

RESULTS_BASE = '../results/modulo_addition_1layer'
FAMILY_NAME  = 'modulo_addition_1layer'
family = load_family(FAMILY_NAME)

VARIANTS = [
    {'label': 'p109/s485/ds598', 'dir': 'modulo_addition_1layer_p109_seed485_dseed598', 'health': 'healthy', 'color': '#1f77b4', 'prime': 109, 'seed': 485, 'dseed': 598},
    {'label': 'p113/s999/ds598', 'dir': 'modulo_addition_1layer_p113_seed999_dseed598', 'health': 'healthy', 'color': '#2ca02c', 'prime': 113, 'seed': 999, 'dseed': 598},
    {'label': 'p101/s485/ds598', 'dir': 'modulo_addition_1layer_p101_seed485_dseed598', 'health': 'late',    'color': '#ff7f0e', 'prime': 101, 'seed': 485, 'dseed': 598},
    {'label': 'p101/s999/ds598', 'dir': 'modulo_addition_1layer_p101_seed999_dseed598', 'health': 'late',    'color': '#d62728', 'prime': 101, 'seed': 999, 'dseed': 598},
    {'label': 'p59/s485/ds598',  'dir': 'modulo_addition_1layer_p59_seed485_dseed598',  'health': 'degraded','color': '#9467bd', 'prime': 59,  'seed': 485, 'dseed': 598},
]

for v in VARIANTS:
    summary_path = f'{RESULTS_BASE}/{v["dir"]}/variant_summary.json'
    try:
        with open(summary_path) as f:
            vs = json.load(f)
        v['onset']     = vs.get('second_descent_onset_epoch', -1)
        v['eff_xover'] = vs.get('effective_dimensionality_cross_over_epoch', -1)
    except FileNotFoundError:
        v['onset']     = -1
        v['eff_xover'] = -1

SITES       = ['attn_out', 'mlp_out', 'resid_post']
SITE_COLORS = {'attn_out': '#2ca02c', 'mlp_out': '#d62728', 'resid_post': '#9467bd'}
SITE_LABELS = {'attn_out': 'Attn', 'mlp_out': 'MLP', 'resid_post': 'Resid Post'}

def load_repr_geometry(variant_dir):
    loader = ArtifactLoader(f'{RESULTS_BASE}/{variant_dir}/artifacts')
    return loader.load_summary('repr_geometry')

repr_data = {v['label']: load_repr_geometry(v['dir']) for v in VARIANTS}

print(f'{"Label":<25} {"onset":>8} {"eff_xover":>10}')
print('-' * 46)
for v in VARIANTS:
    print(f"{v['label']:<25} {v['onset']:>8} {v['eff_xover']:>10}")

Label                        onset  eff_xover
----------------------------------------------
p109/s485/ds598               3584       5200
p113/s999/ds598               9503      11600
p101/s485/ds598              14045      16000
p101/s999/ds598              13043      15500
p59/s485/ds598               15553      22000


## Cell 1: Metric Helpers

Compute PR₃ and f_top3 from the pca_var fields already in `repr_geometry`.
The total variance T cancels in PR₃, so no raw eigenvalues are needed.

In [2]:
def compute_pr3(f1, f2, f3):
    """Participation ratio of the top 3 eigenvalues. Range [1, 3]."""
    numerator   = (f1 + f2 + f3) ** 2
    denominator = f1**2 + f2**2 + f3**2
    return np.where(denominator > 0, numerator / denominator, 1.0)


def extract_dim_metrics(summary, sites):
    """Return PR₃ and f_top3 per site from a repr_geometry summary dict."""
    result = {}
    for site in sites:
        f1 = summary[f'{site}_pca_var_pc1']
        f2 = summary[f'{site}_pca_var_pc2']
        f3 = summary[f'{site}_pca_var_pc3']
        result[site] = {
            'pr3':    compute_pr3(f1, f2, f3),
            'f_top3': f1 + f2 + f3,
        }
    return result


# Pre-compute for all variants
dim_metrics = {v['label']: extract_dim_metrics(repr_data[v['label']], SITES) for v in VARIANTS}

# Sanity check on canon variant
canon = 'p113/s999/ds598'
m     = dim_metrics[canon]
s     = repr_data[canon]
print(f'Canon ({canon}) — first and last epoch values:')
print(f'  epochs: first={s["epochs"][0]}, last={s["epochs"][-1]}')
for site in SITES:
    pr3 = m[site]['pr3']
    ft  = m[site]['f_top3']
    print(f'  {site:<12}  PR₃: {pr3[0]:.3f} → {pr3[-1]:.3f}   f_top3: {ft[0]:.3f} → {ft[-1]:.3f}')

Canon (p113/s999/ds598) — first and last epoch values:
  epochs: first=0, last=24999
  attn_out      PR₃: 2.987 → 2.783   f_top3: 0.184 → 0.798
  mlp_out       PR₃: 2.980 → 2.695   f_top3: 0.118 → 0.695
  resid_post    PR₃: 2.974 → 2.572   f_top3: 0.164 → 0.763


## Cell 2: PR₃ and f_top3 Timeseries — Single Variant

PR₃ (top) and f_top3 (bottom) per site on a shared epoch axis.

Reference lines at PR₃ = 1 (line), 2 (ring), 3 (sphere).

Look for:
- Transient excursions in PR₃ away from 2 (expansion toward 3 or compression toward 1)
- Whether f_top3 moves with PR₃ (expansion within top 3) or inversely (leaking into noise)

In [3]:
FOCUS = 'p113/s999/ds598'
focus_v = next(v for v in VARIANTS if v['label'] == FOCUS)
s   = repr_data[FOCUS]
m   = dim_metrics[FOCUS]
epochs = s['epochs'].tolist()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['PR₃ per site  (1=line · 2=ring · 3=sphere)',
                                    'f_top3 per site  (fraction of total variance in top 3)'])

for site in SITES:
    color = SITE_COLORS[site]
    label = SITE_LABELS[site]
    pr3   = m[site]['pr3'].tolist()
    ft    = m[site]['f_top3'].tolist()

    fig.add_trace(go.Scatter(x=epochs, y=pr3, mode='lines', name=label,
                             legendgroup=site, showlegend=True,
                             line=dict(color=color, width=2.5),
                             hovertemplate=f'{label}<br>Epoch %{{x}}<br>PR₃: %{{y:.3f}}<extra></extra>'),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=ft, mode='lines', name=label,
                             legendgroup=site, showlegend=False,
                             line=dict(color=color, width=2.5),
                             hovertemplate=f'{label}<br>Epoch %{{x}}<br>f_top3: %{{y:.3f}}<extra></extra>'),
                  row=2, col=1)

# Reference lines at PR₃ = 1, 2, 3
for y_val, label_txt in [(1.0, 'line'), (2.0, 'ring'), (3.0, 'sphere')]:
    fig.add_hline(y=y_val, row=1, col=1,
                  line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1),
                  annotation_text=y_val, annotation_position='right')

onset    = focus_v['onset']
eff_xover = focus_v['eff_xover']
for row in [1, 2]:
    if onset > 0:
        fig.add_vline(x=onset, line=dict(color='black', dash='dash', width=1.5), row=row, col=1)
    if eff_xover > 0:
        fig.add_vline(x=eff_xover, line=dict(color='rgba(60,60,210,0.6)', dash='dot', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='f_top3', range=[0, 1.05], row=2, col=1)
fig.update_xaxes(title_text='Epoch', row=2, col=1)
fig.update_layout(
    title=f'{FOCUS} — dimensionality dynamics<br>'
          f'<sup>Dashed: onset ({onset})  ·  Dotted: eff_dim_xover ({eff_xover})</sup>',
    height=650, legend=dict(orientation='h', y=1.03),
)
fig.show()

## Cell 3: Cross-Variant PR₃ Timeseries

One row per variant, all sites overlaid. Shows whether PR₃ behavior is shared across
the health spectrum or specific to individual models.

Reference lines at 1, 2, 3 on each row.

In [4]:
n = len(VARIANTS)
fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=[
                        f"{v['label']} ({v['health']}, onset={v['onset']})"
                        for v in VARIANTS
                    ])

for row_idx, v in enumerate(VARIANTS, 1):
    epochs = repr_data[v['label']]['epochs'].tolist()
    m      = dim_metrics[v['label']]

    for site in SITES:
        color = SITE_COLORS[site]
        pr3   = m[site]['pr3'].tolist()
        fig.add_trace(go.Scatter(
            x=epochs, y=pr3,
            mode='lines', name=SITE_LABELS[site],
            legendgroup=site, showlegend=(row_idx == 1),
            line=dict(color=color, width=2),
            hovertemplate=f"{SITE_LABELS[site]}<br>Epoch %{{x}}<br>PR₃: %{{y:.3f}}<extra></extra>",
        ), row=row_idx, col=1)

    for y_val in [1.0, 2.0, 3.0]:
        fig.add_hline(y=y_val, row=row_idx, col=1,
                      line=dict(color='rgba(0,0,0,0.12)', dash='dot', width=1))

    if v['onset'] > 0:
        fig.add_vline(x=v['onset'], row=row_idx, col=1,
                      line=dict(color='black', dash='dash', width=1.5))

    fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=n, col=1)
fig.update_layout(
    title='PR₃ across health spectrum  (1=line · 2=ring · 3=sphere · dashed=onset)',
    height=230 * n, legend=dict(orientation='h', y=1.02),
)
fig.show()

## Cell 4: f_top3 Cross-Variant

Same layout for f_top3. Shows whether the model concentrates variance in the top 3
components (high, stable) or bleeds into higher dimensions (falling f_top3).

A healthy grokker should show f_top3 rising or staying high through grokking.
A degraded model may show persistent diffusion (low f_top3 throughout).

In [5]:
fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=[
                        f"{v['label']} ({v['health']}, onset={v['onset']})"
                        for v in VARIANTS
                    ])

for row_idx, v in enumerate(VARIANTS, 1):
    epochs = repr_data[v['label']]['epochs'].tolist()
    m      = dim_metrics[v['label']]

    for site in SITES:
        color = SITE_COLORS[site]
        ft    = m[site]['f_top3'].tolist()
        fig.add_trace(go.Scatter(
            x=epochs, y=ft,
            mode='lines', name=SITE_LABELS[site],
            legendgroup=site, showlegend=(row_idx == 1),
            line=dict(color=color, width=2),
            hovertemplate=f"{SITE_LABELS[site]}<br>Epoch %{{x}}<br>f_top3: %{{y:.3f}}<extra></extra>",
        ), row=row_idx, col=1)

    if v['onset'] > 0:
        fig.add_vline(x=v['onset'], row=row_idx, col=1,
                      line=dict(color='black', dash='dash', width=1.5))

    fig.update_yaxes(title_text='f_top3', range=[0, 1.05], row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=n, col=1)
fig.update_layout(
    title='f_top3 across health spectrum  (fraction of variance in top 3 components · dashed=onset)',
    height=230 * n, legend=dict(orientation='h', y=1.02),
)
fig.show()

## Cell 5: Geometry State Space — f_top3 vs PR₃

For each site, plot f_top3 (x) vs PR₃ (y), colored by epoch. This shows the *trajectory*
through the (concentration, shape) space rather than the time evolution of each metric
separately.

Landmarks in state space:
- (high f_top3, PR₃ ≈ 2): clean concentrated ring — healthy final state
- (high f_top3, PR₃ → 3): expansion into 3D while staying concentrated — working space
- (low f_top3, PR₃ ≈ 2): ring-shaped but diffuse background noise
- (low f_top3, PR₃ ≈ 1): collapsed and diffuse — pathological

Loops or spirals in this space indicate oscillatory expansion/contraction events.

In [6]:
FOCUS = 'p113/s999/ds598'
focus_v   = next(v for v in VARIANTS if v['label'] == FOCUS)
s         = repr_data[FOCUS]
m         = dim_metrics[FOCUS]
epochs_arr = s['epochs']
col_scale  = (epochs_arr - epochs_arr.min()) / (epochs_arr.max() - epochs_arr.min() + 1e-9)

onset     = focus_v['onset']
eff_xover = focus_v['eff_xover']

fig = make_subplots(rows=1, cols=len(SITES),
                    subplot_titles=[SITE_LABELS[site] for site in SITES],
                    horizontal_spacing=0.10)

for col_idx, site in enumerate(SITES, 1):
    color  = SITE_COLORS[site]
    ft     = m[site]['f_top3']
    pr3    = m[site]['pr3']
    epochs = epochs_arr.tolist()

    # Full trajectory
    fig.add_trace(go.Scatter(
        x=ft.tolist(), y=pr3.tolist(),
        mode='lines+markers',
        name=SITE_LABELS[site],
        marker=dict(
            size=6,
            color=col_scale.tolist(),
            colorscale='RdYlBu_r',
            cmin=0, cmax=1,
            showscale=(col_idx == len(SITES)),
            colorbar=dict(title='epoch', len=0.7, x=1.02) if col_idx == len(SITES) else None,
        ),
        line=dict(color='rgba(100,100,100,0.25)', width=1),
        customdata=epochs,
        hovertemplate=(
            f'{SITE_LABELS[site]}<br>'
            'epoch %{customdata}<br>'
            'f_top3=%{x:.3f}  PR₃=%{y:.3f}<extra></extra>'
        ),
        showlegend=False,
    ), row=1, col=col_idx)

    # Start marker (epoch 0)
    fig.add_trace(go.Scatter(
        x=[ft[0]], y=[pr3[0]], mode='markers',
        marker=dict(color=color, size=10, symbol='circle-open', line=dict(width=2)),
        showlegend=False,
        hovertemplate=f'{SITE_LABELS[site]} epoch 0<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>',
    ), row=1, col=col_idx)

    # Final marker
    fig.add_trace(go.Scatter(
        x=[ft[-1]], y=[pr3[-1]], mode='markers',
        marker=dict(color=color, size=10, symbol='circle'),
        showlegend=False,
        hovertemplate=f'{SITE_LABELS[site]} final<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>',
    ), row=1, col=col_idx)

    # Onset marker — diamond, black outline
    if onset > 0 and onset in epochs:
        idx = epochs.index(onset)
        fig.add_trace(go.Scatter(
            x=[ft[idx]], y=[pr3[idx]], mode='markers+text',
            marker=dict(color='black', size=12, symbol='diamond',
                        line=dict(color='white', width=1)),
            text=['onset'], textposition='top center',
            textfont=dict(size=9),
            showlegend=(col_idx == 1), name='onset',
            hovertemplate=f'onset (ep {onset})<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>',
        ), row=1, col=col_idx)
    elif onset > 0:
        # Interpolate to nearest epoch
        idx = int(np.argmin(np.abs(epochs_arr - onset)))
        fig.add_trace(go.Scatter(
            x=[ft[idx]], y=[pr3[idx]], mode='markers+text',
            marker=dict(color='black', size=12, symbol='diamond',
                        line=dict(color='white', width=1)),
            text=['onset'], textposition='top center',
            textfont=dict(size=9),
            showlegend=(col_idx == 1), name='onset',
            hovertemplate=f'onset (ep {onset})<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>',
        ), row=1, col=col_idx)

    # Eff-dim crossover marker — triangle, blue
    if eff_xover > 0:
        idx = int(np.argmin(np.abs(epochs_arr - eff_xover)))
        fig.add_trace(go.Scatter(
            x=[ft[idx]], y=[pr3[idx]], mode='markers+text',
            marker=dict(color='rgba(60,60,210,0.9)', size=12, symbol='triangle-up',
                        line=dict(color='white', width=1)),
            text=['eff_xover'], textposition='bottom center',
            textfont=dict(size=9),
            showlegend=(col_idx == 1), name='eff_xover',
            hovertemplate=f'eff_xover (ep {eff_xover})<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>',
        ), row=1, col=col_idx)

    # Ring reference line
    fig.add_hline(y=2.0, row=1, col=col_idx,
                  line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1),
                  annotation_text='ring', annotation_position='right')

    fig.update_xaxes(title_text='f_top3', range=[0, 1.05], row=1, col=col_idx)
    fig.update_yaxes(title_text='PR₃' if col_idx == 1 else '',
                     range=[0.9, 3.1], row=1, col=col_idx)

fig.update_layout(
    title=f'{FOCUS} — geometry state space  (open circle=epoch 0 · filled=final · ◆=onset · ▲=eff_xover)<br>'
          '<sup>Color: early=blue → late=red  ·  Target state: high f_top3, PR₃ ≈ 2</sup>',
    height=500, width=1150,
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

## Cell 7: Cross-Variant State Space

All 5 variants overlaid in the same (f_top3, PR₃) space, one panel per site.
Each variant is a trajectory colored by that variant's health color.
Onset is marked with a diamond on each trajectory.

**What to look for:**
- Do healthy variants converge on the same target state (high f_top3, PR₃ ≈ 2)?
- Do degraded variants take a different path, or reach a different final state?
- Is the separation between health groups visible by onset, or only at the final epoch?

In [7]:
fig = make_subplots(rows=1, cols=len(SITES),
                    subplot_titles=[SITE_LABELS[site] for site in SITES],
                    horizontal_spacing=0.10)

for col_idx, site in enumerate(SITES, 1):
    for v in VARIANTS:
        s_v      = repr_data[v['label']]
        m_v      = dim_metrics[v['label']]
        epochs_v = s_v['epochs']
        ft       = m_v[site]['f_top3']
        pr3      = m_v[site]['pr3']
        color    = v['color']
        label    = v['label']
        onset    = v['onset']

        # Trajectory line
        fig.add_trace(go.Scatter(
            x=ft.tolist(), y=pr3.tolist(),
            mode='lines',
            name=f"{label} ({v['health']})",
            legendgroup=label,
            showlegend=(col_idx == 1),
            line=dict(color=color, width=1.5),
            opacity=0.7,
            hovertemplate=(
                f"{label}<br>"
                'f_top3=%{x:.3f}  PR₃=%{y:.3f}<extra></extra>'
            ),
        ), row=1, col=col_idx)

        # Final state marker
        fig.add_trace(go.Scatter(
            x=[ft[-1]], y=[pr3[-1]], mode='markers',
            marker=dict(color=color, size=9, symbol='circle'),
            showlegend=False, legendgroup=label,
            hovertemplate=f'{label} final<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>',
        ), row=1, col=col_idx)

        # Onset marker
        if onset > 0:
            idx = int(np.argmin(np.abs(epochs_v - onset)))
            fig.add_trace(go.Scatter(
                x=[ft[idx]], y=[pr3[idx]], mode='markers',
                marker=dict(color=color, size=10, symbol='diamond',
                            line=dict(color='white', width=1.5)),
                showlegend=False, legendgroup=label,
                hovertemplate=f'{label} onset (ep {onset})<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>',
            ), row=1, col=col_idx)

    # Ring reference line
    fig.add_hline(y=2.0, row=1, col=col_idx,
                  line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1),
                  annotation_text='ring', annotation_position='right')

    fig.update_xaxes(title_text='f_top3', range=[0, 1.05], row=1, col=col_idx)
    fig.update_yaxes(title_text='PR₃' if col_idx == 1 else '',
                     range=[0.9, 3.1], row=1, col=col_idx)

fig.update_layout(
    title='Cross-variant geometry state space  (line=trajectory · filled=final · ◆=onset)<br>'
          '<sup>Target state: high f_top3, PR₃ ≈ 2  ·  Separation by health visible?</sup>',
    height=500, width=1150,
    legend=dict(orientation='h', y=-0.18),
)
fig.show()

## Cell 8: All-Variant State Space — Full Survey

Scan all 30 variants. Two views:

1. **Trajectory overlay** — all 30 paths through (f_top3, PR₃) space, colored by health group.
   Shows whether healthy/late/degraded models follow structurally different routes.

2. **Final state scatter** — just the endpoint of each trajectory. Direct answer to:
   does health predict where you end up in state space?

In [8]:
from pathlib import Path

HEALTH_COLORS  = {'healthy': '#2ca02c', 'late': '#ff7f0e', 'degraded': '#d62728'}
HEALTH_SYMBOLS = {'healthy': 'circle',  'late': 'diamond',  'degraded': 'x'}

# --- Scan all variants ---
scan_base   = Path(RESULTS_BASE)
all_scan    = []
for d in sorted(scan_base.iterdir()):
    if not d.is_dir():
        continue
    vs_path = d / 'variant_summary.json'
    rg_path = d / 'artifacts' / 'repr_geometry' / 'summary.npz'
    if not vs_path.exists() or not rg_path.exists():
        continue
    with open(vs_path) as f:
        vs = json.load(f)
    perf   = vs.get('performance_classification', {})
    health = perf.get('label', 'unknown') if isinstance(perf, dict) else str(perf)
    health = ('healthy' if 'healthy' in str(health) else
              'late'    if 'late'    in str(health) else
              'degraded')
    onset  = int(vs.get('second_descent_onset_epoch') or -1)
    rg     = np.load(str(rg_path))
    label  = d.name.replace('modulo_addition_1layer_', '')
    metrics = extract_dim_metrics(rg, SITES)
    all_scan.append({'label': label, 'health': health, 'onset': onset,
                     'rg': rg, 'metrics': metrics})

counts = {h: sum(1 for v in all_scan if v['health'] == h) for h in HEALTH_COLORS}
print(f'Loaded {len(all_scan)} variants — {counts}')

# --- Figure 1: Trajectory overlay ---
fig1 = make_subplots(rows=1, cols=len(SITES),
                     subplot_titles=[SITE_LABELS[s] for s in SITES],
                     horizontal_spacing=0.10)

legend_seen = set()
for v in all_scan:
    color  = HEALTH_COLORS[v['health']]
    health = v['health']
    show   = health not in legend_seen
    if show:
        legend_seen.add(health)
    for col_idx, site in enumerate(SITES, 1):
        ft  = v['metrics'][site]['f_top3']
        pr3 = v['metrics'][site]['pr3']
        fig1.add_trace(go.Scatter(
            x=ft.tolist(), y=pr3.tolist(),
            mode='lines',
            name=health, legendgroup=health,
            showlegend=(show and col_idx == 1),
            line=dict(color=color, width=1),
            opacity=0.35,
            hovertemplate=f"{v['label']}<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>",
        ), row=1, col=col_idx)
        # Final state dot
        fig1.add_trace(go.Scatter(
            x=[ft[-1]], y=[pr3[-1]], mode='markers',
            marker=dict(color=color, size=6,
                        symbol=HEALTH_SYMBOLS[health],
                        line=dict(color='white', width=0.8)),
            showlegend=False, legendgroup=health,
            hovertemplate=f"{v['label']} final<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>",
        ), row=1, col=col_idx)

for col_idx, site in enumerate(SITES, 1):
    fig1.add_hline(y=2.0, row=1, col=col_idx,
                   line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1),
                   annotation_text='ring', annotation_position='right')
    fig1.update_xaxes(title_text='f_top3', range=[0, 1.05], row=1, col=col_idx)
    fig1.update_yaxes(title_text='PR₃' if col_idx == 1 else '',
                      range=[0.9, 3.1], row=1, col=col_idx)

fig1.update_layout(
    title=f'All {len(all_scan)} variants — trajectory overlay  (marker = final state)',
    height=480, width=1150,
    legend=dict(orientation='h', y=-0.15),
)
fig1.show()

# --- Figure 2: Final state scatter ---
fig2 = make_subplots(rows=1, cols=len(SITES),
                     subplot_titles=[SITE_LABELS[s] for s in SITES],
                     horizontal_spacing=0.10)

legend_seen = set()
for v in all_scan:
    color  = HEALTH_COLORS[v['health']]
    health = v['health']
    show   = health not in legend_seen
    if show:
        legend_seen.add(health)
    for col_idx, site in enumerate(SITES, 1):
        ft  = v['metrics'][site]['f_top3']
        pr3 = v['metrics'][site]['pr3']
        fig2.add_trace(go.Scatter(
            x=[ft[-1]], y=[pr3[-1]], mode='markers',
            name=health, legendgroup=health,
            showlegend=(show and col_idx == 1),
            marker=dict(color=color, size=9,
                        symbol=HEALTH_SYMBOLS[health],
                        line=dict(color='white', width=1)),
            hovertemplate=f"{v['label']}<br>f_top3=%{{x:.3f}}  PR₃=%{{y:.3f}}<extra></extra>",
        ), row=1, col=col_idx)

for col_idx, site in enumerate(SITES, 1):
    fig2.add_hline(y=2.0, row=1, col=col_idx,
                   line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1),
                   annotation_text='ring', annotation_position='right')
    fig2.update_xaxes(title_text='f_top3', range=[0, 1.05], row=1, col=col_idx)
    fig2.update_yaxes(title_text='PR₃' if col_idx == 1 else '',
                      range=[0.9, 3.1], row=1, col=col_idx)

fig2.update_layout(
    title=f'All {len(all_scan)} variants — final state only  (does health predict position?)',
    height=480, width=1150,
    legend=dict(orientation='h', y=-0.15),
)
fig2.show()

Loaded 40 variants — {'healthy': 22, 'late': 12, 'degraded': 6}


## Cell 6: Shared Timeline — PR₃ + f_top3 + Circularity

The expansion/contraction instrument: three signals on a shared epoch axis.

1. **PR₃ per site** — shape of the centroid cloud in top 3 dims
2. **f_top3 per site** — concentration vs diffusion
3. **Circularity per site** — ring formation (from `repr_geometry`; top-2 only)

If the PR₃ spike (expansion) leads circularity rise (ring stabilization), that's the
working-space handoff: the model expands, reorganizes, then compresses back into the ring.

If all three co-move simultaneously, grokking is a single global event with no detectable
upstream driver separable at this resolution.

In [9]:
FOCUS = 'p113/s999/ds598'
focus_v = next(v for v in VARIANTS if v['label'] == FOCUS)
s   = repr_data[FOCUS]
m   = dim_metrics[FOCUS]
epochs = s['epochs'].tolist()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=[
                        'PR₃ per site  (1=line · 2=ring · 3=sphere)',
                        'f_top3 per site  (variance concentration in top 3)',
                        'Circularity per site  (ring planarity, top-2 only)',
                    ])

for site in SITES:
    color = SITE_COLORS[site]
    label = SITE_LABELS[site]
    pr3   = m[site]['pr3'].tolist()
    ft    = m[site]['f_top3'].tolist()
    circ  = s[f'{site}_circularity'].tolist()

    fig.add_trace(go.Scatter(x=epochs, y=pr3, mode='lines', name=label,
                             legendgroup=site, showlegend=True,
                             line=dict(color=color, width=2.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=ft, mode='lines', name=label,
                             legendgroup=site, showlegend=False,
                             line=dict(color=color, width=2.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=epochs, y=circ, mode='lines', name=label,
                             legendgroup=site, showlegend=False,
                             line=dict(color=color, width=2.5)), row=3, col=1)

for y_val in [1.0, 2.0, 3.0]:
    fig.add_hline(y=y_val, row=1, col=1,
                  line=dict(color='rgba(0,0,0,0.12)', dash='dot', width=1))

onset    = focus_v['onset']
eff_xover = focus_v['eff_xover']
for row in [1, 2, 3]:
    if onset > 0:
        fig.add_vline(x=onset, line=dict(color='black', dash='dash', width=1.5), row=row, col=1)
    if eff_xover > 0:
        fig.add_vline(x=eff_xover, line=dict(color='rgba(60,60,210,0.6)', dash='dot', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='f_top3', range=[0, 1.05], row=2, col=1)
fig.update_yaxes(title_text='Circularity', row=3, col=1)
fig.update_xaxes(title_text='Epoch', row=3, col=1)
fig.update_layout(
    title=f'{FOCUS} — dimensionality expansion/contraction timeline<br>'
          f'<sup>Dashed: onset ({onset})  ·  Dotted: eff_dim_xover ({eff_xover})</sup>',
    height=750, legend=dict(orientation='h', y=1.04),
)
fig.show()

## Cell 9: Transient Frequency Timing vs Attn Circularity Wall

**Hypothesis**: Transient frequencies peak *after* the Attn circularity compression event —
the moment when attention achieves maximum ring planarity. That event is the "wall":
the working space closes, and any frequency that hasn't committed is locked out.

**Metrics**:
- `attn_circ_peak_epoch`: epoch of maximum `attn_out_circularity` (ring formation peak)
- `transient_peak_epoch`: epoch of maximum neuron membership for each transient frequency
- `lead = attn_circ_peak_epoch − transient_peak_epoch`
  - Positive → transient peaked BEFORE the wall (frequency was already fading when ring formed)
  - Negative → transient peaked AFTER the wall (frequency was still building when ring closed)
  - ≈ 0 → coincident — wall and transient peak are the same event

If the hypothesis holds: points should fall **above the diagonal** in the scatter
(transient_peak > attn_circ_peak), and lead values should be negative.

In [10]:
from pathlib import Path

results_base = Path(RESULTS_BASE)

# --- Collect cross-reference data ---
rows = []
for d in sorted(results_base.iterdir()):
    if not d.is_dir():
        continue
    tf_path  = d / 'artifacts' / 'transient_frequency' / 'cross_epoch.npz'
    rg_path  = d / 'artifacts' / 'repr_geometry' / 'summary.npz'
    vs_path  = d / 'variant_summary.json'
    if not (tf_path.exists() and rg_path.exists() and vs_path.exists()):
        continue

    tf = np.load(str(tf_path))
    transient_mask = ~tf['is_final']
    if transient_mask.sum() == 0:
        continue  # no transient frequencies

    rg = np.load(str(rg_path))
    with open(vs_path) as f:
        vs = json.load(f)

    perf   = vs.get('performance_classification', {})
    health = perf.get('label', 'unknown') if isinstance(perf, dict) else str(perf)
    health = ('healthy' if 'healthy' in str(health) else
              'late'    if 'late'    in str(health) else
              'degraded')
    onset  = int(vs.get('second_descent_onset_epoch') or -1)

    attn_circ      = rg['attn_out_circularity']
    epochs_rg      = rg['epochs']
    circ_peak_idx  = int(np.argmax(attn_circ))
    circ_peak_ep   = int(epochs_rg[circ_peak_idx])
    circ_peak_val  = float(attn_circ[circ_peak_idx])

    label = d.name.replace('modulo_addition_1layer_', '')
    transient_freqs  = tf['ever_qualified_freqs'][transient_mask]
    transient_peaks  = tf['peak_epoch'][transient_mask]
    homeless_counts  = tf['homeless_count'][transient_mask]

    for freq, peak_ep, homeless in zip(transient_freqs, transient_peaks, homeless_counts):
        lead = circ_peak_ep - int(peak_ep)
        rows.append({
            'label':         label,
            'health':        health,
            'onset':         onset,
            'freq':          int(freq),
            'transient_peak': int(peak_ep),
            'attn_circ_peak': circ_peak_ep,
            'circ_peak_val':  circ_peak_val,
            'lead':           lead,
            'homeless':       int(homeless),
        })

print(f'{"Variant":<45} {"health":<10} {"freq":>5} {"trans_peak":>11} {"attn_peak":>10} {"lead":>7}  homeless')
print('-' * 100)
for r in sorted(rows, key=lambda x: x['lead']):
    print(f"{r['label']:<45} {r['health']:<10} {r['freq']:>5} {r['transient_peak']:>11} "
          f"{r['attn_circ_peak']:>10} {r['lead']:>+7}  {r['homeless']}")

Variant                                       health      freq  trans_peak  attn_peak    lead  homeless
----------------------------------------------------------------------------------------------------
p59_seed485_dseed999                          degraded       4       24500      14500  -10000  0
p59_seed485_dseed999                          degraded      14       22500      14500   -8000  0
p113_seed999_dseed42                          late          12       26900      21700   -5200  0
p59_seed999_dseed999                          degraded      20       12500       7500   -5000  27
p109_seed485_dseed999                         healthy       40       11100       8500   -2600  42
p113_seed999_dseed999                         degraded      39       15500      12900   -2600  36
p103_seed485_dseed999                         late          34       20500      18900   -1600  0
p101_seed999_dseed598                         late           4       18000      16600   -1400  59
p103_seed999_ds

In [11]:
# --- Figure 1: Scatter — transient peak vs attn circularity peak ---
fig1 = go.Figure()

max_ep = max(max(r['transient_peak'], r['attn_circ_peak']) for r in rows)

# Diagonal (y = x) — simultaneous
fig1.add_trace(go.Scatter(
    x=[0, max_ep], y=[0, max_ep],
    mode='lines', name='simultaneous (y=x)',
    line=dict(color='rgba(0,0,0,0.25)', dash='dot', width=1.5),
    showlegend=True,
))

for health, color, symbol in [('healthy', '#2ca02c', 'circle'),
                                ('late',    '#ff7f0e', 'diamond'),
                                ('degraded','#d62728', 'x')]:
    subset = [r for r in rows if r['health'] == health]
    if not subset:
        continue
    fig1.add_trace(go.Scatter(
        x=[r['attn_circ_peak'] for r in subset],
        y=[r['transient_peak']  for r in subset],
        mode='markers',
        name=health,
        marker=dict(color=color, size=11, symbol=symbol,
                    line=dict(color='white', width=1)),
        text=[f"{r['label']}<br>freq {r['freq']}<br>lead {r['lead']:+d}" for r in subset],
        hovertemplate='%{text}<extra></extra>',
    ))

fig1.update_layout(
    title='Transient frequency peak epoch vs Attn circularity peak epoch<br>'
          '<sup>Above diagonal → transient peaked AFTER attn wall  ·  '
          'Below → peaked BEFORE wall</sup>',
    xaxis_title='Attn circularity peak epoch  (the "wall")',
    yaxis_title='Transient frequency peak epoch',
    xaxis=dict(range=[0, max_ep * 1.05]),
    yaxis=dict(range=[0, max_ep * 1.05]),
    height=550, width=700,
    legend=dict(x=0.02, y=0.97),
)
fig1.show()

# --- Figure 2: Lead distribution ---
fig2 = go.Figure()
for health, color in [('healthy', '#2ca02c'), ('late', '#ff7f0e'), ('degraded', '#d62728')]:
    subset = [r['lead'] for r in rows if r['health'] == health]
    if not subset:
        continue
    fig2.add_trace(go.Box(
        y=subset, name=health,
        marker_color=color,
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        hovertemplate='lead: %{y:+d}<extra></extra>',
    ))

fig2.add_hline(y=0, line=dict(color='black', dash='dot', width=1.5),
               annotation_text='simultaneous', annotation_position='right')
fig2.update_layout(
    title='Lead distribution by health  (lead = attn_circ_peak − transient_peak)<br>'
          '<sup>Negative → transient peaked AFTER the attn wall</sup>',
    yaxis_title='Lead (epochs)',
    height=450, width=500,
)
fig2.show()

# Summary
print(f'\nSummary by health group:')
for health in ['healthy', 'late', 'degraded']:
    leads = [r['lead'] for r in rows if r['health'] == health]
    if not leads:
        continue
    arr = np.array(leads)
    n_after = (arr < 0).sum()
    print(f'  {health:<10}: n={len(arr)}  mean_lead={arr.mean():+.0f}  '
          f'median={np.median(arr):+.0f}  peaked_after_wall={n_after}/{len(arr)}')


Summary by health group:
  healthy   : n=7  mean_lead=+3057  median=-300  peaked_after_wall=4/7
  late      : n=5  mean_lead=-900  median=-1400  peaked_after_wall=4/5
  degraded  : n=7  mean_lead=-2886  median=-2600  peaked_after_wall=6/7


## Direction 1: PR₃/f_top3 in Weight Space and Trajectory Domains

The previous sections apply PR₃/f_top3 to **activation-space class centroids** (repr_geometry).
Two new domains for the same lens:

### 1A — Weight-Space Centroid Geometry

`freq_group_weight_geometry` stores `Win_centroids` — the centroid of each frequency group
in W_in space at every epoch. PCA of those n_groups × d_model centroids gives PR₃ in
weight space at each epoch.

**Question:** Does the frequency group centroid ring in weight space form on the same schedule
as the class centroid ring in activation space? Does weight-space ring formation *lead* or *lag*
the activation-space signal?

Note: with 4 groups (typical), max rank = 3, so f_top3 ≈ 1.0 always. PR₃ alone carries
the signal here: 1 = groups collinear, 2 = groups form a ring, 3 = groups in a volume.

### 1B — Parameter Trajectory Rolling Shape

`parameter_trajectory` stores `{site}__projections` — the trajectory of the parameter
vector projected onto global PCs across checkpoints. Rolling variance of PC1/PC2/PC3
coordinates gives the **local shape of the learning path** at each moment.

- **PR₃ ≈ 1**: the model is moving along one axis — directed, committed motion
- **PR₃ ≈ 2**: motion is distributed across a plane — reorganization in progress
- **PR₃ ≈ 3**: motion spread across all three PCs — diffuse, searching

If grokking involves a directed compression, trajectory PR₃ should drop toward 1
at the second descent. If it involves planar reorganization, PR₃ stays near 2.

In [12]:
from pathlib import Path as _Path

def compute_centroid_pr3_timeseries(wg, weight_type='Win'):
    """PR₃ and f_top3 of freq group centroid cloud in weight space.

    wg: dict from freq_group_weight_geometry cross_epoch.npz
    weight_type: 'Win' or 'Wout'
    Returns: (epochs, pr3_array, f_top3_array)

    Note: with n_groups <= 4, rank <= 3, so f_top3 ≈ 1.0 always.
    PR₃ is the informative metric: 1=groups collinear, 2=groups form ring, 3=groups in volume.
    """
    centroids = wg[f'{weight_type}_centroids']   # (n_epochs, n_groups, d)
    epochs    = wg['epochs']
    n_epochs, n_groups, _ = centroids.shape

    pr3_arr   = np.zeros(n_epochs)
    f_top3_arr = np.zeros(n_epochs)

    for t in range(n_epochs):
        C = centroids[t].astype(np.float64)
        C = C - C.mean(axis=0)
        if n_groups < 2:
            pr3_arr[t] = 1.0
            f_top3_arr[t] = 1.0
            continue
        _, s, _ = np.linalg.svd(C, full_matrices=False)
        total_var = (s ** 2).sum()
        if total_var < 1e-12:
            pr3_arr[t] = 1.0
            f_top3_arr[t] = 0.0
            continue
        top_k = min(3, len(s))
        f = (s[:top_k] ** 2) / total_var
        f1 = f[0] if top_k > 0 else 0.0
        f2 = f[1] if top_k > 1 else 0.0
        f3 = f[2] if top_k > 2 else 0.0
        pr3_arr[t]    = compute_pr3(f1, f2, f3)
        f_top3_arr[t] = f1 + f2 + f3

    return epochs, pr3_arr, f_top3_arr


def compute_rolling_trajectory_pr3(projections, window=10):
    """Rolling PR₃ of trajectory in PC space.

    projections: (n_epochs, n_pcs) — PC coordinates from parameter_trajectory
    window: rolling window size (in snapshot steps)

    For each epoch t, measures how the model's motion is distributed across
    PC1/PC2/PC3: PR₃≈1 = directed (single axis), PR₃≈2 = planar, PR₃≈3 = volumetric.
    """
    n_epochs = projections.shape[0]
    proj_3   = projections[:, :3]
    pr3_arr  = np.full(n_epochs, np.nan)

    for t in range(n_epochs):
        start = max(0, t - window + 1)
        window_data = proj_3[start:t + 1]
        if window_data.shape[0] < 2:
            continue
        var = window_data.var(axis=0)   # variance of each PC in window
        total = var.sum()
        if total < 1e-12:
            pr3_arr[t] = 1.0
            continue
        f1, f2, f3 = var / total
        pr3_arr[t] = compute_pr3(f1, f2, f3)

    return pr3_arr


# --- Load weight_geometry for all reference variants ---
wg_data = {}
for v in VARIANTS:
    wg_path = f'{RESULTS_BASE}/{v["dir"]}/artifacts/freq_group_weight_geometry/cross_epoch.npz'
    try:
        wg_data[v['label']] = dict(np.load(wg_path))
    except FileNotFoundError:
        print(f'Warning: no weight_geometry artifact for {v["label"]}')

print(f'Loaded weight_geometry for {len(wg_data)} variants')
for label, wg in wg_data.items():
    n_ep, n_grp, d = wg['Win_centroids'].shape
    freqs = wg['group_freqs'].tolist()
    print(f'  {label}: {n_ep} epochs, {n_grp} groups {freqs}, d={d}')

Loaded weight_geometry for 5 variants
  p109/s485/ds598: 131 epochs, 3 groups [3, 13, 26], d=128
  p113/s999/ds598: 95 epochs, 4 groups [8, 32, 37, 54], d=128
  p101/s485/ds598: 135 epochs, 4 groups [14, 27, 34, 48], d=128
  p101/s999/ds598: 184 epochs, 4 groups [34, 40, 42, 43], d=128
  p59/s485/ds598: 116 epochs, 2 groups [4, 20], d=128


In [13]:
n = len(VARIANTS)
fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=[
                        f"{v['label']} ({v['health']}, onset={v['onset']})"
                        for v in VARIANTS
                    ])

for row_idx, v in enumerate(VARIANTS, 1):
    wg = wg_data[v['label']]
    epochs_wg = wg['epochs'].tolist()

    _, win_pr3,  _ = compute_centroid_pr3_timeseries(wg, 'Win')
    _, wout_pr3, _ = compute_centroid_pr3_timeseries(wg, 'Wout')

    for pr3, wlabel, dash in [(win_pr3, 'W_in', 'solid'), (wout_pr3, 'W_out', 'dash')]:
        fig.add_trace(go.Scatter(
            x=epochs_wg, y=pr3.tolist(),
            mode='lines', name=wlabel,
            legendgroup=wlabel, showlegend=(row_idx == 1),
            line=dict(color='#e377c2' if wlabel == 'W_in' else '#8c564b',
                      width=2, dash=dash),
            hovertemplate=f"{v['label']} {wlabel}<br>Epoch %{{x}}<br>PR₃: %{{y:.3f}}<extra></extra>",
        ), row=row_idx, col=1)

    for y_val in [1.0, 2.0, 3.0]:
        fig.add_hline(y=y_val, row=row_idx, col=1,
                      line=dict(color='rgba(0,0,0,0.12)', dash='dot', width=1))
    if v['onset'] > 0:
        fig.add_vline(x=v['onset'], row=row_idx, col=1,
                      line=dict(color='black', dash='dash', width=1.5))

    fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=n, col=1)
fig.update_layout(
    title='Weight-space centroid PR₃ — cross-variant  (1=line · 2=ring · 3=sphere · dashed=onset)<br>'
          '<sup>Solid: W_in  ·  Dashed: W_out  ·  Groups: neurons by dominant frequency</sup>',
    height=230 * n,
    legend=dict(orientation='h', y=1.02),
)
fig.show()

In [14]:
FOCUS = 'p113/s999/ds598'
focus_v  = next(v for v in VARIANTS if v['label'] == FOCUS)
wg_canon = wg_data[FOCUS]
rg_canon = repr_data[FOCUS]

epochs_wg = wg_canon['epochs'].tolist()
epochs_rg = rg_canon['epochs'].tolist()

# Weight-space centroid PR₃ (W_in and W_out)
_, win_pr3,  _ = compute_centroid_pr3_timeseries(wg_canon, 'Win')
_, wout_pr3, _ = compute_centroid_pr3_timeseries(wg_canon, 'Wout')

# Activation-space centroid PR₃ — use resid_post as it integrates both streams
rp_pr3 = dim_metrics[FOCUS]['resid_post']['pr3']
at_pr3 = dim_metrics[FOCUS]['attn_out']['pr3']
ml_pr3 = dim_metrics[FOCUS]['mlp_out']['pr3']

# Trajectory rolling PR₃ (window = 10 snapshots ≈ 1000 epochs)
pt_canon = np.load(f'{RESULTS_BASE}/{focus_v["dir"]}/artifacts/parameter_trajectory/cross_epoch.npz')
traj_pr3 = {
    site: compute_rolling_trajectory_pr3(pt_canon[f'{site}__projections'], window=10)
    for site in ['all', 'embedding', 'attention', 'mlp']
}
traj_epochs = pt_canon['epochs'].tolist()

# -- 3-row shared timeline --
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=[
                        'Activation-space centroid PR₃  (repr_geometry sites)',
                        'Weight-space centroid PR₃  (W_in and W_out group centroids)',
                        'Trajectory rolling PR₃  (local shape of learning path, window=10)',
                    ])

# Row 1: activation-space PR₃
for (pr3, site_label, color) in [
    (rp_pr3, 'Resid Post', '#9467bd'),
    (at_pr3, 'Attn',       '#2ca02c'),
    (ml_pr3, 'MLP',        '#d62728'),
]:
    fig.add_trace(go.Scatter(x=epochs_rg, y=pr3.tolist(), mode='lines', name=site_label,
                             legendgroup=site_label, showlegend=True,
                             line=dict(color=color, width=2.5)), row=1, col=1)

# Row 2: weight-space PR₃
for (pr3, wlabel, color) in [
    (win_pr3,  'W_in',  '#e377c2'),
    (wout_pr3, 'W_out', '#8c564b'),
]:
    fig.add_trace(go.Scatter(x=epochs_wg, y=pr3.tolist(), mode='lines', name=wlabel,
                             legendgroup=wlabel, showlegend=True,
                             line=dict(color=color, width=2.5)), row=2, col=1)

# Row 3: rolling trajectory PR₃
traj_colors = {'all': '#7f7f7f', 'embedding': '#17becf', 'attention': '#2ca02c', 'mlp': '#d62728'}
for site, pr3 in traj_pr3.items():
    fig.add_trace(go.Scatter(x=traj_epochs, y=pr3.tolist(), mode='lines', name=f'traj {site}',
                             legendgroup=f'traj_{site}', showlegend=True,
                             line=dict(color=traj_colors[site], width=2, dash='dash'
                                       if site == 'all' else 'solid')), row=3, col=1)

# Reference lines and markers
onset     = focus_v['onset']
eff_xover = focus_v['eff_xover']
for row in [1, 2, 3]:
    for y_val in [1.0, 2.0, 3.0]:
        fig.add_hline(y=y_val, row=row, col=1,
                      line=dict(color='rgba(0,0,0,0.10)', dash='dot', width=1))
    if onset > 0:
        fig.add_vline(x=onset, row=row, col=1,
                      line=dict(color='black', dash='dash', width=1.5))
    if eff_xover > 0:
        fig.add_vline(x=eff_xover, row=row, col=1,
                      line=dict(color='rgba(60,60,210,0.6)', dash='dot', width=1.5))

fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=2, col=1)
fig.update_yaxes(title_text='PR₃', range=[0.9, 3.1], row=3, col=1)
fig.update_xaxes(title_text='Epoch', row=3, col=1)
fig.update_layout(
    title=f'{FOCUS} — PR₃ across three domains<br>'
          f'<sup>Row 1: activation-space centroids  ·  Row 2: weight-space centroids  '
          f'·  Row 3: trajectory local shape  ·  Dashed: onset · Dotted: eff_xover</sup>',
    height=850,
    legend=dict(orientation='h', y=1.04),
)
fig.show()

## Direction 1 — Sanity Check: Cross-Variant Trajectory Rolling PR₃

Six-variant sample to test whether the trajectory PR₃ dip toward 1 at onset is a consistent
healthy-model signature or an artifact of the canon variant.

Sample spans the health spectrum:
- p109 / p113-ds598 (healthy, different onset timings)
- p89 (healthy, late onset — no-event smooth variant)
- p101 (late grokker)
- p59 (late grokker, slow)
- p113-ds999 (degraded)

If the dip is real: healthy models show a transient PR₃ → 1 event at onset (directed compression),
degraded models don't. Trajectory site split (`mlp` vs `attention`) may reveal which component drives it.

In [15]:
SAMPLE = [
    {'label': 'p109/s485/ds598', 'dir': 'modulo_addition_1layer_p109_seed485_dseed598', 'health': 'healthy',  'onset': 3584,  'color': '#1f77b4'},
    {'label': 'p113/s999/ds598', 'dir': 'modulo_addition_1layer_p113_seed999_dseed598', 'health': 'healthy',  'onset': 9503,  'color': '#2ca02c'},
    {'label': 'p89/s485/ds598',  'dir': 'modulo_addition_1layer_p89_seed485_dseed598',  'health': 'healthy',  'onset': 11748, 'color': '#17becf'},
    {'label': 'p101/s485/ds598', 'dir': 'modulo_addition_1layer_p101_seed485_dseed598', 'health': 'late',     'onset': 14045, 'color': '#ff7f0e'},
    {'label': 'p59/s485/ds598',  'dir': 'modulo_addition_1layer_p59_seed485_dseed598',  'health': 'late',     'onset': 15553, 'color': '#d62728'},
    {'label': 'p113/s999/ds999', 'dir': 'modulo_addition_1layer_p113_seed999_dseed999', 'health': 'degraded', 'onset': 10048, 'color': '#9467bd'},
]

TRAJ_SITES   = ['mlp', 'attention', 'all']
TRAJ_DASHES  = {'mlp': 'solid', 'attention': 'dash', 'all': 'dot'}
TRAJ_WEIGHTS = {'mlp': 2.5, 'attention': 2.0, 'all': 1.5}
TRAJ_COLORS  = {'mlp': '#d62728', 'attention': '#2ca02c', 'all': '#7f7f7f'}

n = len(SAMPLE)
fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=[
                        f"{v['label']} ({v['health']}, onset={v['onset']})"
                        for v in SAMPLE
                    ])

for row_idx, v in enumerate(SAMPLE, 1):
    pt = np.load(f'{RESULTS_BASE}/{v["dir"]}/artifacts/parameter_trajectory/cross_epoch.npz')
    epochs = pt['epochs'].tolist()

    for site in TRAJ_SITES:
        pr3 = compute_rolling_trajectory_pr3(pt[f'{site}__projections'], window=10)
        fig.add_trace(go.Scatter(
            x=epochs, y=pr3.tolist(),
            mode='lines', name=site,
            legendgroup=site, showlegend=(row_idx == 1),
            line=dict(color=TRAJ_COLORS[site], width=TRAJ_WEIGHTS[site],
                      dash=TRAJ_DASHES[site]),
            hovertemplate=f"{v['label']} {site}<br>Epoch %{{x}}<br>PR₃: %{{y:.3f}}<extra></extra>",
        ), row=row_idx, col=1)

    for y_val in [1.0, 2.0, 3.0]:
        fig.add_hline(y=y_val, row=row_idx, col=1,
                      line=dict(color='rgba(0,0,0,0.12)', dash='dot', width=1))
    if v['onset'] > 0:
        fig.add_vline(x=v['onset'], row=row_idx, col=1,
                      line=dict(color='black', dash='dash', width=1.5))
    fig.update_yaxes(title_text='PR₃', range=[0.8, 3.2], row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=n, col=1)
fig.update_layout(
    title='Trajectory rolling PR₃ — 6-variant sample  (window=10 snapshots ≈ 1000 epochs)<br>'
          '<sup>Solid=MLP  ·  Dashed=Attention  ·  Dotted=All  ·  Vertical dashed=onset</sup>',
    height=240 * n,
    legend=dict(orientation='h', y=1.02),
)
fig.show()

## Minimum Weight-Space PR₃ — All-Variant Scalar Metric

For each variant, find the minimum W_in centroid PR₃ value during training. This is the deepest
point of the weight-space ring compression event.

**Hypothesis:** Healthy models dip well below 2.5 (groups organize toward ring/line in weight
space). Degraded models never dip — they stay near 3.0 throughout.

The minimum is computed over epochs after the first 10% of training (to skip initialization
noise). Window constraint: uses onset ± padding if available, otherwise global minimum.

A clean separation here would mean weight-space centroid PR₃ is a tunable lens on grokking
quality — potentially more robust than onset-based metrics since it doesn't require a precisely
detected onset epoch.

In [16]:
from pathlib import Path as _Path2

results_base = _Path2(RESULTS_BASE)

scalar_rows = []
for d in sorted(results_base.iterdir()):
    if not d.is_dir():
        continue
    wg_path = d / 'artifacts' / 'freq_group_weight_geometry' / 'cross_epoch.npz'
    vs_path = d / 'variant_summary.json'
    if not (wg_path.exists() and vs_path.exists()):
        continue

    wg = dict(np.load(str(wg_path)))
    with open(vs_path) as f:
        vs = json.load(f)

    perf   = vs.get('performance_classification', {})
    health = perf.get('label', 'unknown') if isinstance(perf, dict) else str(perf)
    health = ('healthy' if 'healthy' in str(health) else
              'late'    if 'late'    in str(health) else
              'degraded')
    onset       = int(vs.get('second_descent_onset_epoch') or -1)
    final_loss  = float(vs.get('final_test_loss', np.nan))

    _, win_pr3, _ = compute_centroid_pr3_timeseries(wg, 'Win')
    _, wout_pr3, _ = compute_centroid_pr3_timeseries(wg, 'Wout')
    epochs_wg = wg['epochs']
    n_ep      = len(epochs_wg)

    # Skip first 10% of training (initialization noise)
    skip_idx  = max(1, n_ep // 10)
    win_post  = win_pr3[skip_idx:]
    wout_post = wout_pr3[skip_idx:]

    min_win_pr3  = float(np.nanmin(win_post))
    min_wout_pr3 = float(np.nanmin(wout_post))
    min_win_ep   = int(epochs_wg[skip_idx + int(np.nanargmin(win_post))])

    label = d.name.replace('modulo_addition_1layer_', '')
    scalar_rows.append({
        'label':       label,
        'health':      health,
        'onset':       onset,
        'final_loss':  final_loss,
        'min_win_pr3': min_win_pr3,
        'min_win_ep':  min_win_ep,
        'min_wout_pr3': min_wout_pr3,
        'n_groups':    int(len(wg['group_freqs'])),
    })

scalar_rows.sort(key=lambda r: r['min_win_pr3'])
print(f'{"Variant":<45} {"health":<10} {"onset":>7} {"min_Win_PR₃":>12} {"min_ep":>8} {"n_grp":>6}')
print('-' * 95)
for r in scalar_rows:
    print(f"{r['label']:<45} {r['health']:<10} {r['onset']:>7} "
          f"{r['min_win_pr3']:>12.4f} {r['min_win_ep']:>8} {r['n_groups']:>6}")

Variant                                       health       onset  min_Win_PR₃   min_ep  n_grp
-----------------------------------------------------------------------------------------------
p59_seed485_dseed598                          late         15553       1.0000     1100      2
p109_seed485_dseed598                         healthy       3584       1.1840     5900      3
p107_seed999_dseed598                         healthy       9217       1.2011    17000      3
p59_seed999_dseed598                          healthy       6754       1.2196     9500      3
p109_seed999_dseed598                         healthy      10748       1.2866    12700      4
p59_seed485_dseed42                           late         16022       1.3433    20500      3
p109_seed485_dseed999                         healthy       7373       1.3623    24999      4
p113_seed485_dseed598                         healthy       8676       1.3943    10400      3
p113_seed485_dseed999                         degraded    

In [17]:
# --- Figure 1: min Win PR₃ vs final test loss, colored/shaped by health ---
fig1 = go.Figure()
for health, color, symbol in [
    ('healthy',  '#2ca02c', 'circle'),
    ('late',     '#ff7f0e', 'diamond'),
    ('degraded', '#d62728', 'x'),
]:
    subset = [r for r in scalar_rows if r['health'] == health]
    if not subset:
        continue
    fig1.add_trace(go.Scatter(
        x=[r['min_win_pr3'] for r in subset],
        y=[r['final_loss']   for r in subset],
        mode='markers',
        name=health,
        marker=dict(color=color, size=11, symbol=symbol,
                    line=dict(color='white', width=1)),
        text=[f"{r['label']}<br>onset={r['onset']}<br>min_ep={r['min_win_ep']}"
              for r in subset],
        hovertemplate='%{text}<br>min_Win_PR₃=%{x:.3f}<br>final_loss=%{y:.4f}<extra></extra>',
    ))

fig1.add_vline(x=2.0, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1),
               annotation_text='ring', annotation_position='top right')
fig1.update_layout(
    title='Minimum W_in centroid PR₃ (post-init) vs final test loss<br>'
          '<sup>If healthy models dip lower → weight-space ring compression is a quality signal</sup>',
    xaxis_title='min Win PR₃  (lower = deeper ring formation)',
    yaxis_title='Final test loss',
    yaxis_type='log',
    height=500, width=700,
    legend=dict(x=0.02, y=0.97),
)
fig1.show()

# --- Figure 2: min Win PR₃ vs onset epoch (does early grokking = deeper dip?) ---
fig2 = go.Figure()
for health, color, symbol in [
    ('healthy',  '#2ca02c', 'circle'),
    ('late',     '#ff7f0e', 'diamond'),
    ('degraded', '#d62728', 'x'),
]:
    subset = [r for r in scalar_rows if r['health'] == health and r['onset'] > 0]
    if not subset:
        continue
    fig2.add_trace(go.Scatter(
        x=[r['onset']      for r in subset],
        y=[r['min_win_pr3'] for r in subset],
        mode='markers',
        name=health,
        marker=dict(color=color, size=11, symbol=symbol,
                    line=dict(color='white', width=1)),
        text=[r['label'] for r in subset],
        hovertemplate='%{text}<br>onset=%{x}<br>min_Win_PR₃=%{y:.3f}<extra></extra>',
    ))

fig2.add_hline(y=2.0, line=dict(color='rgba(0,0,0,0.15)', dash='dot', width=1),
               annotation_text='ring', annotation_position='right')
fig2.update_layout(
    title='Onset epoch vs minimum W_in centroid PR₃<br>'
          '<sup>Does earlier grokking correspond to deeper weight-space compression?</sup>',
    xaxis_title='Onset epoch  (second descent start)',
    yaxis_title='min Win PR₃',
    height=500, width=700,
    legend=dict(x=0.02, y=0.97),
)
fig2.show()

## Direction 1C — Within-Group W_in Point Cloud Dimensionality

Each frequency group is a set of ~80–160 neurons. At each epoch their W_in rows form a
point cloud in 128D weight space. PR₃ and f_top3 of that cloud describe its **shape**:

| Shape   | PR₃  | f_top3      | Description                                    |
|---------|------|-------------|------------------------------------------------|
| Blob    | ≈ 3  | low (< 0.5) | Diffuse cloud — no dominant orientation        |
| Saddle  | ≈ 2  | rising      | Flattened to a plane — two dominant directions |
| Spoke   | ≈ 1  | → 1         | Collapsed to a line — single dominant axis     |

Prior findings (REQ_090) show the trajectory blob → saddle during second descent.
This cell tests whether that transition appears in PR₃/f_top3, times it precisely, and
asks whether different groups (dominant vs transient) transition at different epochs.

Group membership is fixed at the final checkpoint (same convention as weight_geometry).
Computation: per-epoch SVD of (n_members × 128), ~95 epochs × 4–5 groups per variant.

In [18]:
import sys as _sys
if '../src' not in _sys.path:
    _sys.path.insert(0, '../src')
from miscope.analysis.artifact_loader import ArtifactLoader


def compute_group_win_pr3(variant_dir):
    """Per-group W_in point cloud PR₃ and f_top3 timeseries.

    For each epoch: extract each group's neuron rows from W_in, compute
    top-3 SVD, return PR₃ and f_top3 per group per epoch.

    Group membership fixed at final checkpoint (argmax of norm_matrix).

    Returns:
        epochs       int array (n_epochs,)
        pr3          float array (n_epochs, n_groups)
        f_top3       float array (n_epochs, n_groups)
        group_freqs  int array (n_groups,)
        group_sizes  int array (n_groups,)
    """
    loader   = ArtifactLoader(f'{RESULTS_BASE}/{variant_dir}/artifacts')
    wg       = dict(np.load(f'{RESULTS_BASE}/{variant_dir}/artifacts'
                            '/freq_group_weight_geometry/cross_epoch.npz'))
    epochs      = wg['epochs']
    group_freqs = wg['group_freqs']
    n_groups    = len(group_freqs)
    n_epochs    = len(epochs)

    # Rebuild group membership from final-epoch neuron_freq_norm
    norm         = loader.load_epoch('neuron_freq_norm', int(epochs[-1]))
    norm_matrix  = norm['norm_matrix']          # (n_freq, d_mlp)
    dominant_freq = np.argmax(norm_matrix, axis=0)

    group_members = {}
    group_sizes   = []
    for g_idx, freq in enumerate(group_freqs):
        members = np.where(dominant_freq == freq)[0]
        group_members[g_idx] = members
        group_sizes.append(len(members))

    pr3    = np.full((n_epochs, n_groups), np.nan)
    f_top3 = np.full((n_epochs, n_groups), np.nan)

    for ep_idx, epoch in enumerate(epochs):
        snap = loader.load_epoch('parameter_snapshot', int(epoch))
        is_transformer = 'W_E' in snap
        W_in_full = (snap['W_in'].T if is_transformer else snap['W_in']).astype(np.float64)

        for g_idx in range(n_groups):
            members = group_members[g_idx]
            W_g = W_in_full[members]
            W_g = W_g - W_g.mean(axis=0)

            if len(members) < 3:
                pr3[ep_idx, g_idx]    = 1.0
                f_top3[ep_idx, g_idx] = 1.0
                continue

            _, s, _ = np.linalg.svd(W_g, full_matrices=False)
            total   = (s ** 2).sum()
            if total < 1e-12:
                pr3[ep_idx, g_idx]    = 1.0
                f_top3[ep_idx, g_idx] = 0.0
                continue

            top_k = min(3, len(s))
            f     = (s[:top_k] ** 2) / total
            f1    = f[0] if top_k > 0 else 0.0
            f2    = f[1] if top_k > 1 else 0.0
            f3    = f[2] if top_k > 2 else 0.0
            pr3[ep_idx, g_idx]    = compute_pr3(f1, f2, f3)
            f_top3[ep_idx, g_idx] = f1 + f2 + f3

    return epochs, pr3, f_top3, group_freqs, np.array(group_sizes)


# Compute for canon variant
FOCUS    = 'p113/s999/ds598'
focus_v  = next(v for v in VARIANTS if v['label'] == FOCUS)
print(f'Computing within-group W_in PR₃ for {FOCUS} ...')
gpr3_epochs, gpr3, gf3, gfreqs, gsizes = compute_group_win_pr3(focus_v['dir'])
print(f'  groups: {list(zip(gfreqs.tolist(), gsizes.tolist()))}  (freq, n_neurons)')
print(f'  epochs: {len(gpr3_epochs)}, PR₃ range: [{gpr3.min():.3f}, {gpr3.max():.3f}]')

Computing within-group W_in PR₃ for p113/s999/ds598 ...
  groups: [(8, 107), (32, 161), (37, 78), (54, 166)]  (freq, n_neurons)
  epochs: 95, PR₃ range: [2.303, 2.999]


In [19]:
GROUP_PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

onset     = focus_v['onset']
eff_xover = focus_v['eff_xover']
epochs_rg = repr_data[FOCUS]['epochs'].tolist()
circ_attn = repr_data[FOCUS]['attn_out_circularity'].tolist()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=[
                        'Within-group W_in PR₃  (blob≈3 · saddle≈2 · spoke≈1)',
                        'Within-group W_in f_top3  (variance concentration in top 3 dims)',
                        'Attn circularity  (ring planarity — reference signal)',
                    ])

for g_idx in range(len(gfreqs)):
    freq  = int(gfreqs[g_idx])
    n     = int(gsizes[g_idx])
    color = GROUP_PALETTE[g_idx % len(GROUP_PALETTE)]
    label = f'freq {freq} (n={n})'

    fig.add_trace(go.Scatter(
        x=gpr3_epochs.tolist(), y=gpr3[:, g_idx].tolist(),
        mode='lines', name=label,
        legendgroup=label, showlegend=True,
        line=dict(color=color, width=2.5),
        hovertemplate=f'{label}<br>Epoch %{{x}}<br>PR₃: %{{y:.3f}}<extra></extra>',
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=gpr3_epochs.tolist(), y=gf3[:, g_idx].tolist(),
        mode='lines', name=label,
        legendgroup=label, showlegend=False,
        line=dict(color=color, width=2.5),
        hovertemplate=f'{label}<br>Epoch %{{x}}<br>f_top3: %{{y:.3f}}<extra></extra>',
    ), row=2, col=1)

fig.add_trace(go.Scatter(
    x=epochs_rg, y=circ_attn,
    mode='lines', name='Attn circularity',
    legendgroup='circ', showlegend=True,
    line=dict(color='#2ca02c', width=2.5, dash='dash'),
), row=3, col=1)

for row in [1, 2, 3]:
    if onset > 0:
        fig.add_vline(x=onset, row=row, col=1,
                      line=dict(color='black', dash='dash', width=1.5))
    if eff_xover > 0:
        fig.add_vline(x=eff_xover, row=row, col=1,
                      line=dict(color='rgba(60,60,210,0.6)', dash='dot', width=1.5))

for y_val in [1.0, 2.0, 3.0]:
    fig.add_hline(y=y_val, row=1, col=1,
                  line=dict(color='rgba(0,0,0,0.12)', dash='dot', width=1))

fig.update_yaxes(title_text='PR₃',    range=[0.9, 3.1], row=1, col=1)
fig.update_yaxes(title_text='f_top3', range=[0, 1.05],  row=2, col=1)
fig.update_yaxes(title_text='Circularity',               row=3, col=1)
fig.update_xaxes(title_text='Epoch', row=3, col=1)
fig.update_layout(
    title=f'{FOCUS} — within-group W_in point cloud dimensionality<br>'
          f'<sup>Per-group PR₃ and f_top3 vs attn circularity  ·  '
          f'Dashed: onset ({onset})  ·  Dotted: eff_xover ({eff_xover})</sup>',
    height=900,
    legend=dict(orientation='v', y=1.04),
)
fig.show()

In [20]:
COMPARE_VARIANTS = [
    {'label': 'p109/s485/ds598', 'dir': 'modulo_addition_1layer_p109_seed485_dseed598', 'health': 'healthy',  'onset': 3584,  'note': 'healthy — early onset'},
    {'label': 'p113/s999/ds598', 'dir': 'modulo_addition_1layer_p113_seed999_dseed598', 'health': 'healthy',  'onset': 9503,  'note': 'canon'},
    {'label': 'p89/s485/ds598',  'dir': 'modulo_addition_1layer_p89_seed485_dseed598',  'health': 'healthy',  'onset': 11748, 'note': 'healthy — late onset'},
    {'label': 'p113/s999/ds999', 'dir': 'modulo_addition_1layer_p113_seed999_dseed999', 'health': 'degraded', 'onset': 10048, 'note': 'degraded'},
    {'label': 'p101/s999/ds999', 'dir': 'modulo_addition_1layer_p101_seed999_dseed999', 'health': 'healthy',  'onset': 5037,  'note': 'rebounder — transient freq'},
]

# Pre-compute (canon already done above)
gpr3_cache = {'p113/s999/ds598': (gpr3_epochs, gpr3, gf3, gfreqs, gsizes)}
for v in COMPARE_VARIANTS:
    if v['label'] in gpr3_cache:
        continue
    print(f'Computing {v["label"]} ...')
    gpr3_cache[v['label']] = compute_group_win_pr3(v['dir'])
print('Done.')

Computing p109/s485/ds598 ...
Computing p89/s485/ds598 ...
Computing p113/s999/ds999 ...
Computing p101/s999/ds999 ...
Done.


In [21]:
n_vars = len(COMPARE_VARIANTS)
# 2 rows per variant: f_top3 (top) and PR₃ (bottom), shared x within each pair
# Plus attn circularity overlay on f_top3 row
row_titles = []
for v in COMPARE_VARIANTS:
    row_titles.append(f"{v['label']} ({v['note']}) — f_top3")
    row_titles.append(f"{v['label']} — PR₃")

fig = make_subplots(
    rows=n_vars * 2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.025,
    subplot_titles=row_titles,
)

for v_idx, v in enumerate(COMPARE_VARIANTS):
    row_f = v_idx * 2 + 1  # f_top3 row
    row_p = v_idx * 2 + 2  # PR₃ row

    ep, pr3_v, ft_v, freqs_v, sizes_v = gpr3_cache[v['label']]
    ep_list = ep.tolist()

    # Attn circularity for this variant
    rg_v   = load_repr_geometry(v['dir'])
    circ_v = rg_v['attn_out_circularity'].tolist()
    ep_rg  = rg_v['epochs'].tolist()

    show_legend = (v_idx == 0)

    for g_idx in range(len(freqs_v)):
        freq  = int(freqs_v[g_idx])
        n_mem = int(sizes_v[g_idx])
        color = GROUP_PALETTE[g_idx % len(GROUP_PALETTE)]
        label = f'freq {freq} (n={n_mem})'

        fig.add_trace(go.Scatter(
            x=ep_list, y=ft_v[:, g_idx].tolist(),
            mode='lines', name=label,
            legendgroup=f'g{g_idx}', showlegend=show_legend,
            line=dict(color=color, width=2),
            hovertemplate=f'{label}<br>Ep %{{x}}<br>f_top3: %{{y:.3f}}<extra></extra>',
        ), row=row_f, col=1)

        fig.add_trace(go.Scatter(
            x=ep_list, y=pr3_v[:, g_idx].tolist(),
            mode='lines', name=label,
            legendgroup=f'g{g_idx}', showlegend=False,
            line=dict(color=color, width=2),
            hovertemplate=f'{label}<br>Ep %{{x}}<br>PR₃: %{{y:.3f}}<extra></extra>',
        ), row=row_p, col=1)

    # Attn circularity on f_top3 row (right y-axis feel via opacity)
    fig.add_trace(go.Scatter(
        x=ep_rg, y=circ_v,
        mode='lines', name='Attn circ',
        legendgroup='circ', showlegend=show_legend,
        line=dict(color='rgba(44,160,44,0.45)', width=1.5, dash='dash'),
        yaxis=f'y{row_f}',
        hovertemplate='Attn circ<br>Ep %{x}<br>%{y:.3f}<extra></extra>',
    ), row=row_f, col=1)

    onset = v['onset']
    for row in [row_f, row_p]:
        if onset > 0:
            fig.add_vline(x=onset, row=row, col=1,
                          line=dict(color='black', dash='dash', width=1.5))
    for y_val in [1.0, 2.0, 3.0]:
        fig.add_hline(y=y_val, row=row_p, col=1,
                      line=dict(color='rgba(0,0,0,0.12)', dash='dot', width=1))

    fig.update_yaxes(title_text='f_top3', range=[0, 1.05],  row=row_f, col=1)
    fig.update_yaxes(title_text='PR₃',    range=[0.9, 3.1], row=row_p, col=1)

fig.update_layout(
    title='Within-group W_in point cloud dimensionality — 5-variant comparison<br>'
          '<sup>f_top3 (top) and PR₃ (bottom) per group  ·  '
          'Dashed green = Attn circularity  ·  Dashed black = onset</sup>',
    height=280 * n_vars,
    legend=dict(orientation='v', y=1.02),
)
fig.show()

## Direction 1D — Model Tempo: Global Parameter Trajectory PR₃

The within-group W_in analysis showed **weight configuration dimensionality**: are a group's
neurons arranged as a blob, saddle, or spoke in weight space?

The **parameter trajectory** captures something different: the *shape of the learning path itself*.
Rolling PR₃ of the trajectory projections (PC1/PC2/PC3 coordinates over a sliding window) asks
whether the model is currently moving in a line, plane, or volume through parameter space.

- **PR₃ ≈ 1**: directed compression — the model is making a focused, single-axis move
- **PR₃ ≈ 2**: planar reorganization — motion spread across two axes
- **PR₃ ≈ 3**: volumetric search — diffuse exploration

Weight groups: `embedding`, `attention`, `mlp`, `all` (from the parameter_trajectory artifact).

**Tempo hypothesis**: if trajectory PR₃ dips (directed move), within-group f_top3 rises
(configuration organizes), and attn circularity peaks — all at the same epoch — the model
has a single coordinated compression event. Staggered transitions would imply a causal chain.

In [22]:
TRAJ_SITE_COLORS  = {'embedding': '#17becf', 'attention': '#2ca02c', 'mlp': '#d62728', 'all': '#7f7f7f'}
TRAJ_SITE_DASHES  = {'embedding': 'dot', 'attention': 'dash', 'mlp': 'solid', 'all': 'dash'}
TRAJ_SITE_WEIGHTS = {'embedding': 1.8, 'attention': 2.0, 'mlp': 2.5, 'all': 1.5}


def build_tempo_view(variant_label, variant_dir, onset, eff_xover,
                     first_descent_end=-1, title_note=''):
    """4-row shared-timeline tempo view for one variant.

    Row 1: Trajectory rolling PR₃ per weight group
    Row 2: Within-group W_in f_top3 per frequency group
    Row 3: Activation-space centroid PR₃ (MLP + Resid Post sites)
    Row 4: Attn circularity

    Vertical markers:
      Black dashed   — second descent onset (grokking)
      Orange dotted  — first_descent_window.end_epoch (end of memorization)
      Blue dotted    — eff_dim_crossover (if available)
    """
    pt     = np.load(f'{RESULTS_BASE}/{variant_dir}/artifacts/parameter_trajectory/cross_epoch.npz')
    rg     = load_repr_geometry(variant_dir)
    ep_rg  = rg['epochs'].tolist()
    ep_pt  = pt['epochs'].tolist()

    ep_grp, pr3_grp, ft_grp, freqs_grp, sizes_grp = gpr3_cache[variant_label]
    ep_grp = ep_grp.tolist()

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=[
            'Trajectory rolling PR₃  (shape of learning path per weight group)',
            'Within-group W_in f_top3  (configuration concentration)',
            'Activation-space centroid PR₃  (MLP + Resid Post)',
            'Attn circularity  (ring planarity)',
        ]
    )

    # Row 1 — trajectory rolling PR₃
    for site in ['embedding', 'attention', 'mlp', 'all']:
        tpr3 = compute_rolling_trajectory_pr3(pt[f'{site}__projections'], window=10)
        fig.add_trace(go.Scatter(
            x=ep_pt, y=tpr3.tolist(), mode='lines', name=f'traj {site}',
            legendgroup=f'traj_{site}', showlegend=True,
            line=dict(color=TRAJ_SITE_COLORS[site], width=TRAJ_SITE_WEIGHTS[site],
                      dash=TRAJ_SITE_DASHES[site]),
        ), row=1, col=1)

    # Row 2 — within-group f_top3
    for g_idx in range(len(freqs_grp)):
        freq  = int(freqs_grp[g_idx])
        n_mem = int(sizes_grp[g_idx])
        color = GROUP_PALETTE[g_idx % len(GROUP_PALETTE)]
        fig.add_trace(go.Scatter(
            x=ep_grp, y=ft_grp[:, g_idx].tolist(), mode='lines',
            name=f'freq {freq} (n={n_mem})',
            legendgroup=f'wg_{g_idx}', showlegend=True,
            line=dict(color=color, width=2),
        ), row=2, col=1)

    # Row 3 — activation-space centroid PR₃
    dim_v = extract_dim_metrics(rg, SITES)
    for site, color in [('mlp_out', '#d62728'), ('resid_post', '#9467bd')]:
        fig.add_trace(go.Scatter(
            x=ep_rg, y=dim_v[site]['pr3'].tolist(), mode='lines',
            name=f'repr {SITE_LABELS[site]}',
            legendgroup=f'repr_{site}', showlegend=True,
            line=dict(color=color, width=2),
        ), row=3, col=1)

    # Row 4 — attn circularity
    fig.add_trace(go.Scatter(
        x=ep_rg, y=rg['attn_out_circularity'].tolist(), mode='lines',
        name='Attn circularity', legendgroup='circ', showlegend=True,
        line=dict(color='#2ca02c', width=2.5),
    ), row=4, col=1)

    # Vertical markers
    for row in range(1, 5):
        if first_descent_end > 0:
            fig.add_vline(x=first_descent_end, row=row, col=1,
                          line=dict(color='rgba(210,140,0,0.75)', dash='dot', width=1.5))
        if onset > 0:
            fig.add_vline(x=onset, row=row, col=1,
                          line=dict(color='black', dash='dash', width=1.5))
        if eff_xover and eff_xover > 0:
            fig.add_vline(x=eff_xover, row=row, col=1,
                          line=dict(color='rgba(60,60,210,0.6)', dash='dot', width=1.5))

    for y_val in [1.0, 2.0, 3.0]:
        for row in [1, 3]:
            fig.add_hline(y=y_val, row=row, col=1,
                          line=dict(color='rgba(0,0,0,0.10)', dash='dot', width=1))

    fig.update_yaxes(title_text='traj PR₃',   range=[0.8, 3.2], row=1, col=1)
    fig.update_yaxes(title_text='f_top3',     range=[0, 1.05],  row=2, col=1)
    fig.update_yaxes(title_text='repr PR₃',   range=[0.8, 3.2], row=3, col=1)
    fig.update_yaxes(title_text='Circ',                         row=4, col=1)
    fig.update_xaxes(title_text='Epoch', row=4, col=1)
    fig.update_layout(
        title=f'{variant_label} — model tempo view  {title_note}<br>'
              f'<sup>Orange dotted=first descent end  ·  Black dashed=onset  ·  Blue dotted=eff_xover</sup>',
        height=900, legend=dict(orientation='v', y=1.04),
    )
    return fig


# Canon variant
focus_v = next(v for v in VARIANTS if v['label'] == 'p113/s999/ds598')
fig = build_tempo_view(
    'p113/s999/ds598', focus_v['dir'],
    onset=focus_v['onset'], eff_xover=focus_v['eff_xover'],
    first_descent_end=1200,
    title_note='(canon)',
)
fig.show()

In [23]:
# Cross-variant tempo comparison — p109, canon, degraded
TEMPO_VARIANTS = [
    {'label': 'p109/s485/ds598', 'dir': 'modulo_addition_1layer_p109_seed485_dseed598',
     'onset': 3584,  'eff_xover': -1,  'fd_end': 1200, 'note': 'healthy early'},
    {'label': 'p113/s999/ds598', 'dir': 'modulo_addition_1layer_p113_seed999_dseed598',
     'onset': 9503,  'eff_xover': focus_v['eff_xover'], 'fd_end': 1200, 'note': 'canon'},
    {'label': 'p113/s999/ds999', 'dir': 'modulo_addition_1layer_p113_seed999_dseed999',
     'onset': 10048, 'eff_xover': -1,  'fd_end': 1200, 'note': 'degraded'},
]

for tv in TEMPO_VARIANTS:
    if tv['label'] not in gpr3_cache:
        print(f'Computing {tv["label"]} ...')
        gpr3_cache[tv['label']] = compute_group_win_pr3(tv['dir'])
    fig = build_tempo_view(
        tv['label'], tv['dir'],
        onset=tv['onset'], eff_xover=tv['eff_xover'],
        first_descent_end=tv['fd_end'],
        title_note=f'({tv["note"]})',
    )
    fig.show()

## Dimensionality View — Prototype

Three-panel shared-timeline view. Each panel shows **PR₃ (solid)** and **f_top3 (dashed)**
for every series in that domain. Y-axis is [0, 3.2] for all panels — PR₃ lives in [1, 3]
and f_top3 in [0, 1], so they separate naturally without a secondary axis.

| Panel | Domain | Series |
|-------|--------|--------|
| 1 | Parameter trajectory | embedding / attention / mlp / all |
| 2 | Class centroid (activation space) | attn_out / mlp_out / resid_post |
| 3 | Within-group W_in point cloud | one line per frequency group |

**Vertical markers:** orange dotted = first descent end · black dashed = onset · blue dotted = eff_xover

Read: solid (PR₃) tells *shape* of the representation/trajectory; dashed (f_top3) tells
*concentration* — how much variance is captured by the top 3 dimensions vs diffusing into higher ones.

In [ ]:
def compute_rolling_trajectory_metrics(projections, window=10):
    """Rolling PR₃ and f_top3 of trajectory in PC space.

    PR₃:   shape of motion within top 3 PCs (1=directed · 2=planar · 3=volumetric)
    f_top3: fraction of total motion captured by top 3 PCs vs all available PCs

    f_top3 detects diffusion into higher PCs that PR₃ alone cannot see.
    """
    n_epochs, n_pcs = projections.shape
    pr3_arr = np.full(n_epochs, np.nan)
    ft_arr  = np.full(n_epochs, np.nan)

    for t in range(n_epochs):
        start = max(0, t - window + 1)
        w     = projections[start:t + 1]
        if w.shape[0] < 2:
            continue
        var   = w.var(axis=0)        # variance of each PC in window
        total = var.sum()
        if total < 1e-12:
            pr3_arr[t] = 1.0
            ft_arr[t]  = 1.0
            continue
        ft_arr[t]  = var[:3].sum() / total
        f1, f2, f3 = var[0] / total, var[1] / total, var[2] / total
        pr3_arr[t] = compute_pr3(f1, f2, f3)

    return pr3_arr, ft_arr


def compute_rolling_trajectory_pr3(projections, window=10):
    """Backward-compatible wrapper — returns PR₃ only."""
    pr3, _ = compute_rolling_trajectory_metrics(projections, window=window)
    return pr3


def _add_pr3_f_top3_traces(fig, row, epochs, pr3, ft, color, name, group, show_legend):
    """Add one solid PR₃ trace and one dashed f_top3 trace to a subplot row."""
    fig.add_trace(go.Scatter(
        x=epochs, y=pr3, mode='lines', name=name,
        legendgroup=group, showlegend=show_legend,
        line=dict(color=color, width=2),
        hovertemplate=f'{name}<br>Ep %{{x}}<br>PR₃ %{{y:.3f}}<extra></extra>',
    ), row=row, col=1)
    fig.add_trace(go.Scatter(
        x=epochs, y=ft, mode='lines', name=name,
        legendgroup=group, showlegend=False,
        line=dict(color=color, width=1.5, dash='dash'),
        hovertemplate=f'{name}<br>Ep %{{x}}<br>f_top3 %{{y:.3f}}<extra></extra>',
    ), row=row, col=1)


def _add_timing_markers(fig, n_rows, onset, fd_end, eff_xover):
    """Add vertical timing markers to all rows."""
    for row in range(1, n_rows + 1):
        if fd_end > 0:
            fig.add_vline(x=fd_end, row=row, col=1,
                          line=dict(color='rgba(210,140,0,0.80)', dash='dot', width=1.5))
        if onset > 0:
            fig.add_vline(x=onset, row=row, col=1,
                          line=dict(color='black', dash='dash', width=1.5))
        if eff_xover > 0:
            fig.add_vline(x=eff_xover, row=row, col=1,
                          line=dict(color='rgba(60,60,210,0.6)', dash='dot', width=1.5))


def build_dimensionality_view(variant_label, variant_dir,
                              onset=-1, eff_xover=-1, fd_end=-1):
    """3-panel dimensionality view: trajectory / class centroid / within-group.

    Each panel: solid = PR₃, dashed = f_top3, same color per series.
    Y-axis [0, 3.2] on all panels — PR₃ in [1,3], f_top3 in [0,1], no overlap.
    """
    pt    = np.load(f'{RESULTS_BASE}/{variant_dir}/artifacts/parameter_trajectory/cross_epoch.npz')
    rg    = load_repr_geometry(variant_dir)
    ep_pt = pt['epochs'].tolist()
    ep_rg = rg['epochs'].tolist()
    ep_grp, pr3_grp, ft_grp, freqs_grp, sizes_grp = gpr3_cache[variant_label]
    ep_grp = ep_grp.tolist()

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
        subplot_titles=[
            'Trajectory rolling PR₃ / f_top3  (shape · concentration of learning path)',
            'Class centroid PR₃ / f_top3  (activation-space geometry per site)',
            'Within-group W_in PR₃ / f_top3  (weight-space point cloud per freq group)',
        ]
    )

    # Panel 1 — parameter trajectory
    for site in ['embedding', 'attention', 'mlp', 'all']:
        tpr3, tft = compute_rolling_trajectory_metrics(pt[f'{site}__projections'])
        _add_pr3_f_top3_traces(fig, 1, ep_pt, tpr3.tolist(), tft.tolist(),
                               TRAJ_SITE_COLORS[site], f'traj {site}',
                               f'traj_{site}', show_legend=True)

    # Panel 2 — class centroid
    dim_v = extract_dim_metrics(rg, SITES)
    for site in SITES:
        _add_pr3_f_top3_traces(fig, 2, ep_rg,
                               dim_v[site]['pr3'].tolist(),
                               dim_v[site]['f_top3'].tolist(),
                               SITE_COLORS[site], SITE_LABELS[site],
                               f'repr_{site}', show_legend=True)

    # Panel 3 — within-group weight geometry
    for g_idx in range(len(freqs_grp)):
        color = GROUP_PALETTE[g_idx % len(GROUP_PALETTE)]
        label = f'freq {int(freqs_grp[g_idx])} (n={int(sizes_grp[g_idx])})'
        _add_pr3_f_top3_traces(fig, 3, ep_grp,
                               pr3_grp[:, g_idx].tolist(),
                               ft_grp[:, g_idx].tolist(),
                               color, label, f'wg_{g_idx}', show_legend=True)

    # Reference lines
    for y_val in [1.0, 2.0, 3.0]:
        for row in [1, 2, 3]:
            fig.add_hline(y=y_val, row=row, col=1,
                          line=dict(color='rgba(0,0,0,0.10)', dash='dot', width=1))

    _add_timing_markers(fig, 3, onset, fd_end, eff_xover)

    for row in [1, 2, 3]:
        fig.update_yaxes(range=[0, 3.2], row=row, col=1)
    fig.update_xaxes(title_text='Epoch', row=3, col=1)
    fig.update_layout(
        title=f'{variant_label} — dimensionality view<br>'
              '<sup>Solid=PR₃ · Dashed=f_top3 · '
              'Orange dotted=first descent end · Black dashed=onset · Blue dotted=eff_xover</sup>',
        height=850,
        legend=dict(orientation='v', y=1.04),
    )
    return fig


In [25]:
DIM_VIEW_VARIANTS = [
    {'label': 'p109/s485/ds598', 'dir': 'modulo_addition_1layer_p109_seed485_dseed598',
     'onset': 3584,  'eff_xover': -1,               'fd_end': 1200},
    {'label': 'p113/s999/ds598', 'dir': 'modulo_addition_1layer_p113_seed999_dseed598',
     'onset': 9503,  'eff_xover': focus_v['eff_xover'], 'fd_end': 1200},
    {'label': 'p113/s999/ds999', 'dir': 'modulo_addition_1layer_p113_seed999_dseed999',
     'onset': 10048, 'eff_xover': -1,               'fd_end': 1200},
]

for v in DIM_VIEW_VARIANTS:
    if v['label'] not in gpr3_cache:
        print(f'Computing within-group PR₃ for {v["label"]} ...')
        gpr3_cache[v['label']] = compute_group_win_pr3(v['dir'])
    fig = build_dimensionality_view(
        v['label'], v['dir'],
        onset=v['onset'], eff_xover=v['eff_xover'], fd_end=v['fd_end'],
    )
    fig.show()